http://127.0.0.1:5000/

to open mlflow ui: 
uv run mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000


In [25]:
import pandas as pd 
import numpy as np 

In [26]:
csv_path = r"C:\Users\calvin\Documents\python\studies\mlops mini project\data\LI-Small_Trans.csv"

df_init = pd.read_csv(csv_path)
df_init.head()


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022/09/01 00:21,3402,80021DAD0,3402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0
3,2022/09/01 00:16,3814,8006AD080,3814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,0
4,2022/09/01 00:00,20,8006AD530,20,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,0


In [27]:
def load_data(csv_path: str,sample_size: int = 100_000):
    """
    takes in the expected <size>_trans.csv from IBM AML-SIM Kaggle 
    1. Converts data type from strings into more memory efficient     
    """
    dtypes = {
        "From Bank": "int32",
        "To Bank": "int32",
        "Amount Received": "float32",
        "Receiving Currency": "category",
        "Amount Paid": "float32",
        "Payment Currency": "category",
        "Payment Format": "category",
        "Is Laundering": "int8"
    }

    df = pd.read_csv(csv_path, usecols = list(dtypes.keys()) + ['Account'] + ['Account.1'] +["Timestamp"],dtype = dtypes)

    #make column naming less confusing
    df = df.rename(columns={
        "Account": "sender_account",
        "Account.1": "receiver_account"
    })
    illicit_df = df[df["Is Laundering"] == 1]
    
    # Sample normal transactions
    normal_df = df[df["Is Laundering"] == 0]
    n_samples = min(sample_size, len(normal_df))
    normal_sample = normal_df.sample(n=n_samples, random_state=42)

    # Combine and sort strictly by Timestamp
    sampled_df = pd.concat([illicit_df, normal_sample], axis=0)
    sampled_df["Timestamp"] = pd.to_datetime(sampled_df["Timestamp"])
    sampled_df = sampled_df.sort_values(by="Timestamp").reset_index(drop=True)

    return sampled_df

    return df 

In [28]:
df = load_data(csv_path, sample_size = 200000)
df.shape

(203565, 11)

In [29]:
df.describe() 

,Timestamp,From Bank,To Bank,Amount Received,Amount Paid,Is Laundering
count,203565,203565.000000,203565.000000,2.035650e+05,2.035650e+05,203565.000000
mean,2022-09-05 07:53:25.304251904,58885.187901,83944.678904,3.739990e+06,2.445768e+06,0.017513
min,2022-09-01 00:00:00,0.000000,0.000000,1.000000e-06,1.000000e-06,0.000000
25%,2022-09-02 04:56:00,217.000000,11047.000000,1.768100e+02,1.777700e+02,0.000000
50%,2022-09-05 13:05:00,14172.000000,29559.000000,1.423380e+03,1.424710e+03,0.000000
75%,2022-09-08 03:55:00,110265.000000,146606.000000,1.218720e+04,1.211988e+04,0.000000
max,2022-09-17 15:28:00,376910.000000,376754.000000,1.349372e+11,4.761655e+10,1.000000
std,NaN,90066.726096,90668.586828,3.528999e+08,1.383782e+08,0.131172


In [30]:

def preprocess_data(df):
    """
    Performs simple preprocessing and feature engineering 
    1. Generate Temporal Features (Month, Day of the Week)
    2. Generate useful features (Cross currency check, cross bank check, number of transactions within a timeframe to a single acc)
    """

    #generate temporal features
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df['Hour'] = df['Timestamp'].dt.hour.astype('int8')
    df['Day of week'] = df['Timestamp'].dt.dayofweek.astype('int8') 

    ## ----------------this is a place to implement feature store -> talk about this-----------------------------------------

    #feature engineering 
    #1. cross currency check 
    df['Cross currency check'] = (df['Receiving Currency'] != df['Payment Currency']).astype('int8')

    #2. Cross bank check 
    df['Cross bank check'] = (df['From Bank'] != df['To Bank']).astype('int8')

    #3. Number of transactions to an account within timeframe (1h and 24h) 
    df = df.sort_values(["receiver_account", "Timestamp"]).reset_index(drop=True)
    rolled = (
        df.groupby("receiver_account", group_keys=False)
        .rolling("1h", on="Timestamp")["Amount Received"]
        .agg(["count", "sum"])
    )
    df["received_count_1h"] = rolled["count"].to_numpy().astype("int32")
    df["received_amt_1h"] = rolled["sum"].to_numpy().astype("float64")
    # same for 24h, then:
    rolled = (
        df.groupby("receiver_account", group_keys=False)
        .rolling("24h", on="Timestamp")["Amount Received"]
        .agg(["count", "sum"])
    )
    df["received_count_24h"] = rolled["count"].to_numpy().astype("int32")
    df["received_amt_24h"] = rolled["sum"].to_numpy().astype("float64")

    df = df.sort_values("Timestamp").reset_index(drop=True)

    return df


In [31]:
df = preprocess_data(df)
df

,Timestamp,From Bank,sender_account,To Bank,receiver_account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,Hour,Day of week,Cross currency check,Cross bank check,received_count_1h,received_amt_1h,received_count_24h,received_amt_24h
0,2022-09-01 00:00:00,12592,80150AF30,12592,80150AF30,1582.160034,US Dollar,1582.160034,US Dollar,Reinvestment,0,0,3,0,0,1,1582.160034,1,1582.160034
1,2022-09-01 00:00:00,25400,818172280,25400,818172280,7.160000,Euro,7.160000,Euro,Reinvestment,0,0,3,0,0,1,7.160000,1,7.160000
2,2022-09-01 00:00:00,36975,80E2FE470,36975,80E2FE470,5497.370117,Canadian Dollar,5497.370117,Canadian Dollar,Reinvestment,0,0,3,0,0,1,5497.370117,1,5497.370117
3,2022-09-01 00:00:00,232406,8144C87C0,22129,803028BC0,5.660000,US Dollar,5.660000,US Dollar,Credit Card,0,0,3,0,1,1,5.660000,1,5.660000
4,2022-09-01 00:00:00,264772,8181EEB10,264772,8181EEB10,89.010002,Saudi Riyal,89.010002,Saudi Riyal,Reinvestment,0,0,3,0,0,1,89.010002,1,89.010002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203560,2022-09-16 12:10:00,11,8001AB0D0,23319,8013FA900,6921.450195,US Dollar,6921.450195,US Dollar,ACH,1,12,4,0,1,1,6921.450195,1,6921.450195
203561,2022-09-16 12:41:00,1291,8001F2990,12,8001F8B00,8515.969727,US Dollar,8515.969727,US Dollar,ACH,1,12,4,0,1,1,8515.969727,1,8515.969727
203562,2022-09-16 13:24:00,11,8001AB0D0,1439,8011BC1F0,14709.160156,US Dollar,14709.160156,US Dollar,ACH,1,13,4,0,1,1,14709.160156,1,14709.160156
203563,2022-09-17 02:32:00,1291,8001F2990,3,80023D080,13053.019531,Euro,13053.019531,Euro,ACH,1,2,5,0,1,1,13053.019531,1,13053.019531


In [32]:
df[(df['received_count_1h'] >= 5)]

df[(df['received_count_1h'] != df['received_count_24h'])]

df[(df['received_count_1h'] >= 1) & (df['Is Laundering'] == 1)]




,Timestamp,From Bank,sender_account,To Bank,receiver_account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,Hour,Day of week,Cross currency check,Cross bank check,received_count_1h,received_amt_1h,received_count_24h,received_amt_24h
248,2022-09-01 00:00:00,11968,815630C40,249349,815635220,8.923300e+02,US Dollar,8.923300e+02,US Dollar,ACH,1,0,3,0,1,1,8.923300e+02,1,8.923300e+02
678,2022-09-01 00:01:00,70,10042B660,11305,807861770,1.097976e+06,US Dollar,1.097976e+06,US Dollar,Cash,1,0,3,0,1,1,1.097976e+06,1,1.097976e+06
953,2022-09-01 00:02:00,70,10042B660,22661,805F7F2B0,7.083164e+04,US Dollar,7.083164e+04,US Dollar,Cash,1,0,3,0,1,1,7.083164e+04,1,7.083164e+04
1497,2022-09-01 00:03:00,70,10042B6A8,212854,806242CD0,2.247874e+04,Euro,2.247874e+04,Euro,Cash,1,0,3,0,1,1,2.247874e+04,1,2.247874e+04
1669,2022-09-01 00:03:00,70,10042B6F0,111615,8045CDF10,1.485613e+08,Yuan,1.485613e+08,Yuan,Cash,1,0,3,0,1,1,1.485613e+08,1,1.485613e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203560,2022-09-16 12:10:00,11,8001AB0D0,23319,8013FA900,6.921450e+03,US Dollar,6.921450e+03,US Dollar,ACH,1,12,4,0,1,1,6.921450e+03,1,6.921450e+03
203561,2022-09-16 12:41:00,1291,8001F2990,12,8001F8B00,8.515970e+03,US Dollar,8.515970e+03,US Dollar,ACH,1,12,4,0,1,1,8.515970e+03,1,8.515970e+03
203562,2022-09-16 13:24:00,11,8001AB0D0,1439,8011BC1F0,1.470916e+04,US Dollar,1.470916e+04,US Dollar,ACH,1,13,4,0,1,1,1.470916e+04,1,1.470916e+04
203563,2022-09-17 02:32:00,1291,8001F2990,3,80023D080,1.305302e+04,Euro,1.305302e+04,Euro,ACH,1,2,5,0,1,1,1.305302e+04,1,1.305302e+04


In [33]:
import time
import mlflow
import mlflow.lightgbm
import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
)
mlflow.set_tracking_uri("sqlite:///mlflow.db")  # same URI as the UI
mlflow.set_experiment("aml-detection")

feature_cols = [
    "Amount Received", "Amount Paid",
    "Hour", "Day of week",
    "Cross currency check", "Cross bank check",
    "received_count_1h", "received_amt_1h",
    "received_count_24h", "received_amt_24h",
]
cat_cols = ["Receiving Currency", "Payment Currency", "Payment Format"]

X = df[feature_cols + cat_cols].copy()
y = df["Is Laundering"]
for c in cat_cols:
    X[c] = X[c].astype("category")

# chronological 70 / 15 / 15
n = len(df)
i_train, i_val = int(n * 0.70), int(n * 0.85)
X_train, y_train = X.iloc[:i_train], y.iloc[:i_train]
X_val, y_val = X.iloc[i_train:i_val], y.iloc[i_train:i_val]
X_test, y_test = X.iloc[i_val:], y.iloc[i_val:]

n_pos, n_neg = int((y_train == 1).sum()), int((y_train == 0).sum())
scale_pos_weight = n_neg / max(n_pos, 1)

params = {
    "objective": "binary",
    "metric": "average_precision",  # PR-AUC, not accuracy
    "learning_rate": 0.01,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 100,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": scale_pos_weight,
    "n_estimators": 300,
    "random_state": 42,
    "verbosity": -1,
}

with mlflow.start_run(run_name="lgbm-baseline-v2"):
    mlflow.log_params(params)
    mlflow.log_param("n_train", len(X_train))
    mlflow.log_param("pos_rate_train", float(y_train.mean()))
    mlflow.log_param("scale_pos_weight", scale_pos_weight)

    t0 = time.perf_counter()
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_X = X_val,
        eval_y = y_val,
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
    )
    train_s = time.perf_counter() - t0
    mlflow.log_metric("train_seconds", train_s)

    def log_split(name, Xs, ys):
        t1 = time.perf_counter()
        proba = model.predict_proba(Xs)[:, 1]
        infer_ms = (time.perf_counter() - t1) / max(len(Xs), 1) * 1000 #inference time per row in miliseconds
        pr_auc = average_precision_score(ys, proba)
        roc = roc_auc_score(ys, proba)
        mlflow.log_metrics({
            f"{name}_pr_auc": pr_auc,
            f"{name}_roc_auc": roc,
            f"{name}_infer_ms_per_row": infer_ms,
        })
        return proba, pr_auc

    log_split("val", X_val, y_val)
    _, test_pr = log_split("test", X_test, y_test)

    mlflow.lightgbm.log_model(model, name="model", input_example=X_train.head(5))
    print("test PR-AUC:", test_pr)

Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.216626
[100]	valid_0's average_precision: 0.232883
[150]	valid_0's average_precision: 0.230088
Early stopping, best iteration is:
[109]	valid_0's average_precision: 0.235868


2026/08/31 23:48:29 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\calvin\Documents\python\studies\mlops mini project
c:\Users\calvin\Documents\python\studies\mlops mini project\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/08/31 23:48:29 WARNING mlflow.mode

test PR-AUC: 0.3492720046633651


Once we have registered the model on MLFLow 
(to do this, go to models, click on model name, register model, set name) -> registered models are typically what we deploy to production. so now subsequent inference can jusst call that model 

The extra codes below is to set a signature to the MLFlow model. A signature is metadata, and it specifies the expected schema of the input and output of our model so that no errors during inference. 
Without signature, the model is just guessing what the output of the inference should be, and is susceptible to failure

the code below extracts the model from load_model by the saved name (-v1/1), 

the logged_model_uri is the newly logged model that we get from running the code ABOVE, which we logged the model as "model". 

Note: logged and registered doesnt mean the same thing! All of the training runs from the code above will log models to MLFlow, but the registering is separate, can be done manually in the UI or via code as seen below

Once we retrieved the logged model, we want to set a signature for it, to allow it to be informed about the expected schema.

Then, we use mlflow.register_model to register this model, and the name. If the name already exists, like in this case, then it will automatically increment the version in mlflow. 

If we look at the models in MLFlow, we see that the first model version 1, does not have any input output schema defined. The second model implemented the signature which resolved this. 

The extra codes below is to set a signature to the MLFlow model. A signature is metadata, and it specifies the expected schema of the input and output of our model so that no errors during inference. 
Without signature, the model is just guessing what the output of the inference should be, and is susceptible to failure

the code below extracts the model from load_model by the saved name (-v1/1), 

the logged_model_uri is the newly logged model that we get from running the code ABOVE, which we logged the model as "model". 

Note: logged and registered doesnt mean the same thing! All of the training runs from the code above will log models to MLFlow, but the registering is separate, can be done manually in the UI or via code as seen below

Once we retrieved the logged model, we want to set a signature for it, to allow it to be informed about the expected schema.

Then, we use mlflow.register_model to register this model, and the name. If the name already exists, like in this case, then it will automatically increment the version in mlflow. 



In [ ]:
from mlflow.models import infer_signature, set_signature 

mlflow.set_tracking_uri("sqlite:///mlflow.db")



model = mlflow.lightgbm.load_model("models:/aml-lgbm-v1/1")
logged_model_uri = "models:/m-c5d25951d02f46d6b4fb50d8df6c1bc3"

X_example = X_train.head(5).copy() 
proba = model.predict_proba(X_example)[:, 1]
X_sig = X_example.copy()
for c in ["Receiving Currency", "Payment Currency", "Payment Format"]:
    X_sig[c] = X_sig[c].astype(str)

signature = infer_signature(X_sig, proba)
#registered models are immutable, so we cannot set_signature to the model directly. we have to 
#re-access the run uri, set signature, then re-register the model
set_signature(logged_model_uri, signature) 
mlflow.register_model(logged_model_uri, "aml-lgbm-v1")




c:\Users\calvin\Documents\python\studies\mlops mini project\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'aml-lgbm-v1' already exists. Creating a new version of this model...
Created version '2' of model 'aml-lgbm-v1'.


<ModelVersion: aliases=[], creation_timestamp=1788191552097, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1788191552097, metrics=None, model_id=None, name='aml-lgbm-v1', params=None, run_id='456846a0c54e41f4834e24119306c95b', run_link=None, source='models:/m-c5d25951d02f46d6b4fb50d8df6c1bc3', status='READY', status_message=None, tags={}, user_id=None, version=2, workspace='default'>